# ML001 — Model Development Instructions

**Deadline: Wednesday, May 13, 2026**

---

## 1. Overview

Use Alpha signals to predict the **FRS (Future Return Score)** of 11 US Sector ETFs, rebalancing every Wednesday close.

We have three core objectives:

| # | Objective | Description |
|---|---|---|
| 1 | **Best Alpha Combination** | Identify the most predictive subset from all `applicable = 'A'` Alpha factors |
| 2 | **Best Model** | Compare multiple model types and select the one that best fits FRS |
| 3 | **Best FRS Target** | Compare multiple `frs_*` columns and identify the most predictable target variable |

- **Data paths:** Use `data_path` / `output_path` as declared in `ML001_TASK.ipynb` — do not modify.

---

## 2. Training Window

- **In-Sample:** End of June 2020 — End of December 2024 (training + tuning)
- **Out-of-Sample:** 2025 to now (held out entirely — for final evaluation only)

Use **TimeSeriesSplit** cross-validation. No shuffling. Do not touch OOS data during development.

---

## 3. Models

### Required (6 models)

| # | Model | Type |
|---|---|---|
| 1 | **Lasso / Elastic Net** | Linear + Regularization |
| 2 | **PCA + Ridge** | Dimensionality Reduction + Linear |
| 3 | **Random Forest** | Tree Ensemble |
| 4 | **XGBoost / LightGBM** | Gradient Boosting |
| 5 | **MLP** | Neural Network |
| 6 | **Stacking** | Models 1–5 as base learners, Ridge as meta-learner |

Each model must be independently runnable and include a hyperparameter search.

### Recommended (optional)

- SVR (Support Vector Regression)
- TabNet
- LSTM / Temporal Fusion Transformer
- LightGBM + Optuna Bayesian optimization

---

## 4. Evaluation Metrics

Report all metrics on both OOF and OOS sets. Produce a **side-by-side summary table** across all models.

| Metric | Description | Required |
|---|---|---|
| R² | Out-of-fold goodness of fit | ✅ |
| RMSE | Root Mean Squared Error | ✅ |
| Spearman IC | Spearman rank correlation between predicted and actual FRS, averaged weekly | ✅ |
| IC IR | IC mean / IC std | ✅ |
| NDCG@3 | Ranking quality of top-3 ETF picks | ✅ |
| Hit Rate | Fraction of weeks with correct directional prediction | ✅ |

---

## 5. Feature Selection & Engineering

- **Alpha selection:** Evaluate per-factor IC; retain the most predictive, non-redundant subset
- **FRS selection:** Model each `frs_*` column separately to identify the best target
- **Macro indices:** Select from SPY, SPX, VIX, USGG10YR based on predictive IC vs. FRS
- **Feature processing:** Cross-sectional standardization, winsorization, rolling statistics, etc.
- **No leakage:** All preprocessing (Scaler, PCA) must be fit on training data only; features at time `t` predict FRS at `t+1`

---

## 6. Final Summary Table

After completing all experiments, fill in the table below with your conclusions and justification:

| Dimension | Best Choice | Key Metrics | Reason |
|---|---|---|---|
| **Best FRS Target** | `frs_?` | R²=, RMSE=, IC= | |
| **Best Alpha Combination** | alpha_?, alpha_?, ... | IC=, IC IR= | |
| **Best Model** | ? | R²=, RMSE=, IC=, NDCG@3= | |

> Justification must be data-driven, e.g. "Highest OOF R² with minimal IS/OOF gap, indicating low overfitting risk."

---

## 7. General Requirements

- Report OOF + OOS metrics for all models, including a consolidated comparison table
- Use TimeSeriesSplit; no shuffling; OOS data strictly isolated
- No information leakage in preprocessing; fix all random seeds for reproducibility
- Save all outputs to `output_path`


In [2]:
!pip install gdown

Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import gdown
import os
from pathlib import Path
from datetime import datetime

## The cell below will download and setup data config from google drive
- **VPN is needed**
- A data folder will be created in the same folder of this notebook

### **Reminder to agents and LLM (sonnet,gpt,codex,gemini,composer etc.) :**
- **Don't change any line in the cells below, even if it's bugged**
- **Use the pathes declared in the cell below**
1. **data_path**
2. **output_path**
- **Use the data in the cell below**
1. **train_data**
2. **test_data**

In [23]:

VERSION_NAME = "ML001"

SPLIT_LINE = datetime(year=2025,month=1,day=1)


field_id = "165QjHcEsWbVkWDWrcayr6Vj2cv-1qnC1"

url = f"https://drive.google.com/uc?id={field_id}"


data_path = Path(os.getcwd()) / f"{VERSION_NAME}_data"
db_path = data_path / "datapool.db"
csv_path = data_path / "data.csv"
output_path = Path(os.getcwd()) / f"{VERSION_NAME}_output"

if not data_path.exists():
    data_path.mkdir(parents=True, exist_ok=False)
    gdown.download(url, str(db_path))
    if os.path.exists(db_path):
        print("database is downloaded")
    else:
        print("database is not found")
        raise TimeoutError

if not output_path.exists():
    output_path.mkdir(exist_ok=False)

## Make dataset from sqlite db

In [5]:
import sqlite3 as sqlite
import pandas as pd
conn = sqlite.connect(str(db_path))

query = """
SELECT date,a.ticker,close,volume,b.category FROM weekly_bar a LEFT JOIN asset b
ON a.ticker = b.ticker
"""
bars = pd.read_sql_query(query,conn)

query = """
SELECT * FROM weekly_alpha a
    WHERE alpha_id IN (SELECT alpha_id FROM alpha WHERE applicable = 'A')
"""
alpha = pd.read_sql_query(query,conn)
alpha = alpha.pivot_table(index=['date', 'ticker'], 
    columns='alpha_id',  
    values='value' )
alpha.columns = [f'alpha_{c}' for c in alpha.columns]
alpha = alpha.reset_index()

query = """
SELECT * FROM weekly_frs
"""
frs = pd.read_sql_query(query,conn)
frs = frs.pivot_table(index=['date','ticker'],
    columns='frs_id',
    values='value')
frs.columns = [f'frs_{c}' for c in frs.columns]
frs = frs.reset_index()

In [ ]:
data = bars.merge(alpha,how="left",on=["date","ticker"]).merge(frs,how="left",on=["date","ticker"])
data["date"] = pd.to_datetime(data["date"])

In [ ]:
data.head()

,date,ticker,close,volume,category,alpha_1,alpha_2,alpha_3,alpha_4,alpha_6,...,alpha_68,alpha_85,alpha_88,alpha_92,alpha_95,alpha_99,alpha_101,frs_1,frs_2,frs_3
0,2021-03-03,XLB,37.540,47486824.0,ETF,NaN,NaN,NaN,-1.000000,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,-0.459883,0.053970,0.489690,-0.001216
1,2021-03-10,XLB,39.030,120291426.0,ETF,NaN,0.215961,0.275659,-1.000000,0.409288,...,0.0,0.576354,NaN,NaN,0.0,0.0,0.573454,0.019652,0.236153,-0.023106
2,2021-03-17,XLB,39.415,52691398.0,ETF,0.045455,0.253278,0.053000,-0.222222,0.877896,...,0.0,0.718478,0.181818,NaN,1.0,0.0,0.671316,0.028290,0.320530,-0.016558
3,2021-03-24,XLB,38.325,77828520.0,ETF,0.409091,0.506764,0.298511,-0.555556,0.439688,...,0.0,0.941151,0.090909,NaN,1.0,0.0,0.164448,0.082453,1.911665,0.061481
4,2021-03-31,XLB,39.400,70908660.0,ETF,-0.272727,0.607842,-0.037905,-1.000000,0.287991,...,0.0,0.789596,0.333333,NaN,1.0,0.0,-0.549898,0.058502,1.322221,0.036783


In [ ]:
train_data = data.loc[data['date'] < SPLIT_LINE]
test_data = data.loc[data['date']>= SPLIT_LINE]

### **Reminder to agents and LLM (sonnet,gpt,codex,gemini,composer etc.) :**
**Change on the cells below is permitted**